# Delta table Query util
This notebook uses **Polars SQL** to query the Delta tables in the Delta lake tables from S3. 

It maintains compatibility with the original SQL syntax by automatically resolving `schema.table` references to the correct S3 Delta paths.

In [1]:
import os
import re

import certifi
import polars as pl

from electricity_maps.config import get_settings

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir
so = settings.storage_options

In [2]:
def get_sql_df(sql_query):
    """Execute SQL query using Polars SQLContext."""
    ctx = pl.SQLContext()

    # 1. Identify all 'schema.table' references (e.g. bronze.electricity_flows)
    table_refs = re.findall(r'([a-zA-Z0-9_]+\.[a-zA-Z0-9_]+)', sql_query)

    # 2. Register each table in the context and prepare a version of the SQL
    # that Polars' parser can handle (identifiers with dots usually need quoting)
    working_sql = sql_query
    for ref in set(table_refs):
        schema, table = ref.split('.')
        path = f"{DATA_DIR}/{schema}/{table}"

        try:
            # Use scan_delta for efficient lazy execution
            lf = pl.scan_delta(path, storage_options=so)

            # Register the table. We use a safe alias (no dots) for the internal registration
            # to avoid namespace parsing issues in Polars SQL.
            safe_alias = f"{schema}_{table}"
            ctx.register(safe_alias, lf)

            # Replace the original 'schema.table' with the 'safe_alias' in the working SQL
            working_sql = working_sql.replace(ref, safe_alias)

        except Exception as e:
            print(f"Warning: Could not load table {ref} from {path}: {e}")

    # 3. Execute the modified SQL and collect results
    return ctx.execute(working_sql).collect()

def print_sql_df(sql_query, label=None):
    """Helper to display the result of a SQL query."""
    if label:
        print(f"\n{label}:")
    return get_sql_df(sql_query)

In [11]:
print_sql_df('select * from bronze.electricity_flows order by process_ts desc limit 10', "Bronze Daily Flows Summary")


Bronze Daily Flows Summary:


process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,year,month,day
i64,"datetime[μs, UTC]",str,str,"datetime[μs, UTC]","datetime[μs, UTC]",str,i64,i64,i64,i64
1777301860571,2026-04-27 14:57:40.571618 UTC,"""https://api.electricitymaps.co…","""FR""",2026-04-27 13:00:00 UTC,2026-04-27 14:00:00 UTC,"""{""zone"": ""FR"", ""temporalGranul…",1,2026,4,27
1777223123493,2026-04-26 17:05:23.493984 UTC,"""https://api.electricitymaps.co…","""FR""",2026-04-26 13:00:00 UTC,2026-04-26 17:00:00 UTC,"""{""zone"": ""FR"", ""temporalGranul…",4,2026,4,26


In [12]:
print_sql_df('select * from bronze.electricity_mix order by process_ts desc limit 10', "Bronze Daily Mix Summary")


Bronze Daily Mix Summary:


process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,year,month,day
i64,"datetime[μs, UTC]",str,str,"datetime[μs, UTC]","datetime[μs, UTC]",str,i64,i64,i64,i64
1777301860571,2026-04-27 14:57:40.571618 UTC,"""https://api.electricitymaps.co…","""FR""",2026-04-27 13:00:00 UTC,2026-04-27 14:00:00 UTC,"""{""zone"": ""FR"", ""temporalGranul…",1,2026,4,27
1777223123493,2026-04-26 17:05:23.493984 UTC,"""https://api.electricitymaps.co…","""FR""",2026-04-26 13:00:00 UTC,2026-04-26 17:00:00 UTC,"""{""zone"": ""FR"", ""temporalGranul…",4,2026,4,26


In [5]:
print_sql_df('select * from silver.electricity_mix order by process_ts desc limit 6', "Silver Mix")


Silver Mix:


process_ts,zone,datetime,updated_at,is_estimated,estimation_method,nuclear_mw,geothermal_mw,biomass_mw,coal_mw,wind_mw,solar_mw,hydro_mw,gas_mw,oil_mw,unknown_mw,hydro_storage_charge_mw,hydro_storage_discharge_mw,battery_storage_charge_mw,battery_storage_discharge_mw,flow_exports_mw,flow_imports_mw,year,month,day
i64,str,datetime[μs],datetime[μs],bool,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32
1777223153884,"""FR""",2026-04-26 13:00:00,2026-04-26 16:10:07.341,true,"""SANDBOX_MODE_DATA""",23223.345,null,928.774,0.0,573.591,10752.647,2564.09,260.748,40.678,null,2132.566,null,null,55.957,10443.0,100.0,2026,4,26
1777223153884,"""FR""",2026-04-26 14:00:00,2026-04-26 16:10:07.341,true,"""SANDBOX_MODE_DATA""",34026.659,null,1420.101,0.0,514.181,10038.885,3252.378,403.698,37.449,null,2553.868,null,null,36.084,9376.0,0.0,2026,4,26
1777223153884,"""FR""",2026-04-26 15:00:00,2026-04-26 16:39:27.255,true,"""SANDBOX_MODE_DATA""",28114.943,null,1052.044,0.0,904.815,8261.493,3854.653,388.716,39.197,null,2278.633,null,null,58.605,7960.0,242.0,2026,4,26
1777223153884,"""FR""",2026-04-26 16:00:00,2026-04-26 16:39:27.255,true,"""SANDBOX_MODE_DATA""",30448.361,null,983.667,0.0,609.967,10484.144,3709.205,354.58,29.799,null,2851.521,null,null,13.048,10939.0,0.0,2026,4,26
1777212885259,"""FR""",2026-04-26 12:00:00,2026-04-26 13:55:18.001,true,"""SANDBOX_MODE_DATA""",23198.216,null,929.576,0.0,569.769,11423.678,2518.921,271.613,40.678,null,2303.251,null,12.585,null,9378.0,570.0,2026,4,26
1777206207772,"""FR""",2026-04-26 11:00:00,2026-04-26 11:54:21.220,true,"""SANDBOX_MODE_DATA""",27653.723,null,1071.846,0.0,560.688,9640.912,3118.09,435.373,43.521,null,2834.244,null,13.286,null,7922.0,1082.0,2026,4,26


In [6]:
print_sql_df('select * from silver.electricity_flows order by process_ts desc limit 10', "Silver Flows")


Silver Flows:


process_ts,zone,datetime,updated_at,neighbor_zone,direction,power_mw,year,month,day
i64,str,datetime[μs],datetime[μs],str,str,f64,i32,i32,i32
1777223153884,"""FR""",2026-04-26 13:00:00,2026-04-26 16:10:07.341,"""BE""","""import""",2365.0,2026,4,26
1777223153884,"""FR""",2026-04-26 13:00:00,2026-04-26 16:10:07.341,"""DE""","""import""",1028.0,2026,4,26
1777223153884,"""FR""",2026-04-26 13:00:00,2026-04-26 16:10:07.341,"""ES""","""import""",2666.0,2026,4,26
1777223153884,"""FR""",2026-04-26 13:00:00,2026-04-26 16:10:07.341,"""GB""","""export""",3016.0,2026,4,26
1777223153884,"""FR""",2026-04-26 13:00:00,2026-04-26 16:10:07.341,"""IT-NO""","""export""",180.0,2026,4,26
1777223153884,"""FR""",2026-04-26 13:00:00,2026-04-26 16:10:07.341,"""LU""","""export""",127.0,2026,4,26
1777223153884,"""FR""",2026-04-26 14:00:00,2026-04-26 16:10:07.341,"""BE""","""import""",2044.0,2026,4,26
1777223153884,"""FR""",2026-04-26 14:00:00,2026-04-26 16:10:07.341,"""CH""","""import""",602.0,2026,4,26
1777223153884,"""FR""",2026-04-26 14:00:00,2026-04-26 16:10:07.341,"""DE""","""import""",1691.0,2026,4,26


In [7]:
print_sql_df('select * from gold.daily_exports order by process_ts desc limit 10', "Gold Daily Exports")


Gold Daily Exports:


process_ts,zone,zone_name,destination_zone,destination_zone_name,date,export_mwh,hours_covered,year,month,day
i64,str,str,str,str,date,f64,i32,i32,i32,i32
1777223204564,"""FR""","""France""","""GB""","""Great Britain""",2026-04-26,52833.0,17,2026,4,26
1777223204564,"""FR""","""France""","""LU""","""LU""",2026-04-26,3455.0,17,2026,4,26
1777223204564,"""FR""","""France""","""IT-NO""","""Italy North""",2026-04-26,13718.0,17,2026,4,26
1777205541513,"""FR""","""France""","""IT-NO""","""Italy North""",2026-04-25,26146.0,12,2026,4,25
1777205541513,"""FR""","""France""","""GB""","""Great Britain""",2026-04-25,36655.0,12,2026,4,25
1777205541513,"""FR""","""France""","""LU""","""LU""",2026-04-25,2315.0,12,2026,4,25


In [8]:
print_sql_df('SELECT * FROM gold.daily_imports order by process_ts desc LIMIT 10', "Gold Daily Imports")


Gold Daily Imports:


process_ts,zone,zone_name,source_zone,source_zone_name,date,import_mwh,hours_covered,year,month,day
i64,str,str,str,str,date,f64,i32,i32,i32,i32
1777223204564,"""FR""","""France""","""DE""","""Germany""",2026-04-26,25831.0,17,2026,4,26
1777223204564,"""FR""","""France""","""ES""","""Spain""",2026-04-26,34489.0,17,2026,4,26
1777223204564,"""FR""","""France""","""CH""","""Switzerland""",2026-04-26,8750.0,16,2026,4,26
1777223204564,"""FR""","""France""","""BE""","""Belgium""",2026-04-26,55254.0,17,2026,4,26
1777205541513,"""FR""","""France""","""ES""","""Spain""",2026-04-25,28389.0,12,2026,4,25
1777205541513,"""FR""","""France""","""CH""","""Switzerland""",2026-04-25,17661.0,12,2026,4,25
1777205541513,"""FR""","""France""","""DE""","""Germany""",2026-04-25,15160.0,12,2026,4,25
1777205541513,"""FR""","""France""","""BE""","""Belgium""",2026-04-25,31998.0,12,2026,4,25


In [9]:
print_sql_df('SELECT * FROM gold.daily_electricity_mix order by process_ts desc LIMIT 10', "Gold Daily Mix")


Gold Daily Mix:


process_ts,zone,zone_name,date,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,oil_pct,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month,day
i64,str,str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32
1777223204564,"""FR""","""France""",2026-04-26,72.671725,2.329597,3.896878,12.37589,7.875096,0.772257,0.078557,0.0,0.0,0.0,820943.463,99.149186,26.477461,17,2026,4,26
1777205541513,"""FR""","""France""",2026-04-25,73.844472,2.209815,5.084784,7.562436,9.987998,1.077564,0.23293,0.0,0.0,0.0,654326.799,98.689506,24.845033,12,2026,4,25


In [10]:
print_sql_df('SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10', "Pipeline State (el_state)")


Pipeline State (el_state):


process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
str,i64,"datetime[μs, UTC]","datetime[μs, UTC]",str,i64,str
"""gold""",1777223204564,2026-04-26 17:06:49.610194 UTC,2026-04-26 17:07:01.854672 UTC,"""R""",8,null
"""silver""",1777223153884,2026-04-26 17:06:06.581631 UTC,2026-04-26 17:06:28.640011 UTC,"""C""",32,null
"""bronze""",1777223123493,2026-04-26 17:05:23.493984 UTC,2026-04-26 17:05:36.179015 UTC,"""C""",8,null
"""bronze""",1777220910307,2026-04-26 16:28:30.307016 UTC,2026-04-26 16:28:34.763617 UTC,"""C""",2,null
"""bronze""",1777220841657,2026-04-26 16:27:21.657825 UTC,2026-04-26 16:27:25.021086 UTC,"""C""",2,null
"""bronze""",1777220785217,2026-04-26 16:26:25.217956 UTC,2026-04-26 16:26:30.034957 UTC,"""C""",2,null
"""bronze""",1777220614789,2026-04-26 16:23:34.789398 UTC,2026-04-26 16:23:39.220870 UTC,"""C""",2,null
"""gold""",1777216802197,2026-04-26 15:20:06.957933 UTC,2026-04-26 15:20:19.564635 UTC,"""R""",8,null
"""silver""",1777216202290,2026-04-26 15:10:08.761385 UTC,2026-04-26 15:10:18.820470 UTC,"""C""",16,null
